# 에이전트 호스팅하기

[노트북 00](./00_The_one_liner_research_agent.ipynb)에서 리서치 에이전트를 만들었습니다. 지금은 여러분의 노트북에서 돌아가죠. 이제 다른 누군가가 그것을 써야 합니다. 동료, 크론 잡, 웹 앱, 고객 같은 사람들이요. 즉 *여러분의 터미널이 아닌 어딘가*에서 돌아가고, 계속 살아 있고, 재시작해도 대화가 유지되고, API 키가 새어 나가지 않아야 합니다.

이 노트북은 똑같은 에이전트를 세 단계로 배포합니다.

| 단계 | 실행 위치 | 언제 쓰나 |
|---|---|---|
| **1. Docker** | 내 머신 / 단일 VM | 개발 루프, 사내 도구, 단일 테넌트 |
| **2. Modal** | 관리형 서버리스 | 인프라 관리 없이 URL과 0까지 축소를 원할 때 |
| **3. Kubernetes** | 자체 클러스터 | 멀티테넌트, 규제 환경, 완전한 통제 |

에이전트 코드, 컨테이너 이미지, HTTP 인터페이스는 세 단계 모두에서 **동일합니다**. 컨테이너를 둘러싼 운영 장치만 달라집니다. 에이전트가 안정된 인터페이스 뒤에 컨테이너로 담기고 나면, 호스트 선택은 다시 쓰는 일이 아니라 배포 결정이 됩니다.

> **이 노트북을 끝까지 실행하는 비용:** Anthropic API 호출에 대략 **$1~2**, Modal 컴퓨트에 **몇 센트** 정도입니다. 모든 단계에 정리 절차가 있습니다.

모든 배포 코드는 이 노트북 옆의 [`hosting/`](./hosting/)에 있습니다.

## 시작하기 전에: Agent SDK가 맞을까요?

**고객 대면 채팅 제품**을 만들고 있다면 [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview)를 먼저 살펴보세요. 호스팅, 세션, UI가 기본 제공되므로 이 노트북의 대부분을 건너뛸 수 있습니다.

Agent SDK는 **프로그램 수준의 통제**가 필요할 때 알맞습니다. 배치나 잡 형태의 에이전트, 사내 도구, 여러분의 백엔드에 내장되는 에이전트, 또는 인프라를 직접 소유해야 하는 규제 환경 같은 경우죠. 해당된다면 계속 읽으세요.

## 머릿속 모형

구분해 둘 명사 셋:

- **프로세스**는 SDK가 적재된 채 실행 중인 Python 인터프리터 하나입니다. Anthropic API와 대화합니다.
- **세션**은 대화 하나입니다. 나중에 `resume=`할 수 있도록 SDK가 디스크에 기록하는 프롬프트 이력, 도구 호출, 결과입니다.
- **컨테이너**는 프로세스와 그 파일 시스템을 묶은 것입니다. 여러분의 에이전트 코드, SDK, Node, 그리고 세션이 놓일 자리입니다.

웹 서버 안에서 "에이전트"를 객체로 생성하는 인프로세스 SDK(OpenAI Agents SDK, Google ADK)와 달리, Claude Agent SDK 에이전트는 그 자체가 **프로세스**입니다. 덕분에 격리가 간단해지지만(컨테이너 하나 = 영향 범위 하나), 호스팅이 `pip install` 문제가 아니라 분산 시스템 문제가 됩니다.

어느 단계의 배포든 같은 네 가지 일을 해야 합니다.

```
┌────────────────────────────────────────────────────────────────────┐
│  caller ──► gateway ──► [ spawn | route ] ──► agent container ──► API
│                 │                                     │
│                 └── auth (not the agent's job)        └── /data  (persist)
│                                                                      │
│  orchestrator ──────────────── lifecycle ────────────────────────────┘
└────────────────────────────────────────────────────────────────────┘
```

1. 작업이 들어오면 컨테이너를 **띄우기**
2. 각 요청을 해당 세션을 가진 컨테이너로 **라우팅하기**
3. **수명 주기**: 유휴 컨테이너를 종료하고, 죽은 것을 재시작하기
4. 재시작해도 대화를 잃지 않도록 세션 기록 **보존하기**

1단계는 네 가지를 모두 손으로 합니다. 2단계는 띄우기와 수명 주기를 Modal에 맡깁니다. 3단계는 네 가지 모두를 Kubernetes와 작은 게이트웨이에 맡깁니다. 에이전트 컨테이너는 전혀 바뀌지 않습니다.

## 배포할 에이전트

노트북 00의 [`research_agent/agent.py`](./research_agent/agent.py), 즉 `WebSearch`와 `Read`를 쓰는 한 줄짜리 리서치 에이전트를 그대로 재사용합니다. 노트북 00을 아직 하지 않았다면 먼저 하세요. 이 노트북은 그 에이전트가 이미 동작한다고 가정합니다.

`hosting/`이 더하는 것은 얇은 HTTP 서버와 Dockerfile뿐입니다. 시스템 프롬프트는 `research_agent.agent`에서 그대로 가져옵니다. 복사하지 않고 임포트합니다. 의도적으로 다른 점이 하나 있습니다. 호스팅 서버는 도구 목록을 `WebSearch` 하나로 좁힙니다. 서버에는 업로드 경로가 없으므로 `Read`가 닿을 수 있는 파일은 다른 세션의 대화 기록과 컨테이너 자신의 환경뿐인데, 프롬프트 인젝션이 섞인 웹 결과가 에이전트를 유도해 그것들을 흘리게 할 수 있습니다. 이 판단의 근거는 `server.py`의 주석에 적어 두었습니다.

In [1]:
from research_agent.agent import DEFAULT_MODEL, RESEARCH_SYSTEM_PROMPT

print(f"model: {DEFAULT_MODEL}")
print(RESEARCH_SYSTEM_PROMPT)

model: claude-opus-4-6
You are a research agent specialized in AI.

When providing research findings:
- Always include source URLs as citations
- Format citations as markdown links: [Source Title](URL)
- Group sources in a "Sources:" section at the end of your response


### 준비

API 키를 담은 `hosting/.env`를 만드세요. 이 파일은 gitignore되어 있습니다.

In [2]:
%%bash
test -f hosting/.env || cp hosting/.env.example hosting/.env
echo 'Edit hosting/.env and set ANTHROPIC_API_KEY, then re-run this cell.'
grep -q '^ANTHROPIC_API_KEY=sk-ant-' hosting/.env \
  && ! grep -q 'your-key-here' hosting/.env \
  && echo '✅ key looks set'

Edit hosting/.env and set ANTHROPIC_API_KEY, then re-run this cell.


✅ key looks set


---
## 1a단계 — 일회성: 프롬프트 하나, 컨테이너 하나, 끝

가능한 가장 단순한 배포입니다. 환경 변수로 받은 프롬프트에 대해 에이전트를 **한 번** 실행하고, 결과를 출력한 뒤 종료하는 컨테이너입니다. 서버도, 세션도, 상태도 없습니다.


> **모델 참고:** 이 노트북을 진행하는 동안 테스트 요청이 저렴하도록 호스팅 계층의 기본값은 `claude-sonnet-4-6`입니다. 노트북 00의 설정과 정확히 맞추려면 `hosting/.env`에 `MODEL=claude-opus-4-6`을 설정하세요.

이것만으로도 실제 업무가 꽤 많이 처리됩니다. 인보이스 처리, 야간 보고서 생성, 일괄 번역, 일회성 분석 같은 것들이죠. 에이전트의 일이 "입력을 받아 출력을 만들고 멈추는 것"이라면 이 절 이후는 필요하지 않습니다.

[`Dockerfile`](./hosting/Dockerfile)은 에이전트, SDK, 그리고 SDK가 내부적으로 구동하는 Claude Code CLI를 담습니다. 이미지가 `research_agent/`와 `utils/`도 필요로 하므로 빌드 컨텍스트는 `hosting/`이 아니라 `claude_agent_sdk/`(이 디렉터리)입니다:

In [3]:
%%bash
docker build -f hosting/Dockerfile -t research-agent . | tail -n 3

#0 building with "orbstack" instance using docker driver

#1 [internal] load build definition from D

ockerfile
#1 transferring dockerfile: 2.30kB done
#1 DONE 0.0s

#2 resolve image config for docker-i

mage://docker.io/docker/dockerfile:1


#2 DONE 0.6s



#3 docker-image://docker.io/docker/dockerfile:1@sha256:87999aa3d42bdc6bea60565083ee17e86d1f3339802f

...


#12 [5/9] RUN pip install --no-cache-dir -r hosting/requirements.txt
#12 CACHED

#13 [8/9] COPY hos

ting/server.py hosting/run_once.py hosting/entrypoint.sh ./hosting/
#13 CACHED

#14 [9/9] RUN chmod 

+x hosting/entrypoint.sh  && touch hosting/__init__.py
#14 CACHED

#15 exporting to image
#15 export

ing layers done
#15 writing image sha256:b77b20f557cef5e4b9ef01212f3ba3a0895ee97ff8e31f79d9e1dc0cfb7

414f5 done
#15 naming to docker.io/library/research-agent done
#15 DONE 0.0s


In [4]:
%%bash
docker run --rm --env-file hosting/.env \
  -e PROMPT='What is the Claude Agent SDK, in one paragraph?' \
  research-agent

🤖 Thinking...


🤖 Using: ToolSearch()


✓ Tool completed


🤖 Thinking...


🤖 Using: WebSearch()


...

sdk/overview)
- [Building Agents with the Claude Agent SDK – Anthropic Engineering](https://www.an

thropic.com/engineering/building-agents-with-the-claude-agent-sdk)
- [Agent SDK Overview – Anthrop

ic API Docs](https://docs.anthropic.com/en/docs/claude-code/sdk/sdk-overview)
- [claude-agent-sdk-py

thon – GitHub](https://github.com/anthropics/claude-agent-sdk-python)
- [claude-agent-sdk-typescri

pt – GitHub](https://github.com/anthropics/claude-agent-sdk-typescript)


이것으로 끝입니다. `entrypoint.sh`가 `serve` 인자를 받지 못하면 [`run_once.py`](./hosting/run_once.py)가 `$PROMPT`로 `research_agent.agent.send_query()`를 호출하고 0으로 종료합니다.

**이것으로 충분한 경우:** 호출마다 독립적인 잡 형태의 작업입니다. 크론, 큐 워커, CI 단계 등 CLI를 돌릴 수 있는 어디서든 실행하세요.

---
## 1b단계 — 하이브리드: 대화를 이어 갈 수 있도록 서버 붙이기

일회성 모드로는 대화를 이어 갈 수 없습니다. `docker run`마다 새 세션이 시작되니까요. 채팅 형태의 에이전트라면 후속 요청을 받고 매번 알맞은 세션을 재개하는, 오래 살아 있는 프로세스가 필요합니다.

[`hosting/server.py`](./hosting/server.py)는 딱 그 일만 하는 100줄 남짓의 FastAPI 앱입니다. 이 인터페이스가 모든 단계가 따르는 계약입니다.

```
GET  /health                            → 200 {"status": "ok"}
POST /sessions/{session_id}/messages    → 200 text/event-stream of SDK messages
```

`server.py`에서 눈여겨볼 것 두 가지:

- **서버에 인증이 없습니다.** 독스트링에 대놓고 적혀 있습니다. 인증은 게이트웨이의 일입니다(3단계에서 그 자리를 보여 줍니다). 서버는 `session_id` 형식만 검증하고 호출자를 신뢰합니다.
- **여러분의 `session_id`와 SDK 내부 ID를 잇는 작은 매핑을 유지합니다.** SDK는 자기 세션 ID를 스스로 만들며, 여러분이 고를 수 없습니다. 서버는 첫 턴의 `ResultMessage`에서 SDK의 ID를 알아내 후속 요청의 `resume=`에 넘깁니다. 이 매핑은 `/data` 아래 대화 기록 옆에 보존됩니다.

docker-compose로 시작하세요. 재시작해도 대화 기록이 살아남도록 `./sessions`를 `/data`에 마운트하기도 합니다:

In [5]:
%%bash
cd hosting/docker && docker compose up --build -d
sleep 3
curl -s http://localhost:8000/health

 Image research-agent Building 


#1 [internal] load local bake definitions


#1 reading from stdin 579B done
#1 DONE 0.0s



#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 2.30kB done


#2 DONE 0.0s

#3 resolve image config for docker-image://docker.io/docker/dockerfile:1


...

 Container docker-research-agent-1 Creating 


 Container docker-research-agent-1 Created 


 Container docker-research-agent-1 Starting 


 Container docker-research-agent-1 Started 


{"status":"ok"}

프롬프트를 보내고 응답을 스트리밍합니다(`-N`은 curl의 버퍼링을 꺼서 이벤트가 도착하는 대로 보이게 합니다):

In [6]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"What are the three most interesting AI agent trends right now?"}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01LpCYD1mQgNAmiDYq55RHyC", "name": "WebSearch", "input": {"query": "AI agent trends 2026"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, "usage": {"input_tokens": 685, "cache_cr … [truncated]

[... 6 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "Great question! Based on the latest research and reports, here are the **three most interesting AI agent trends** right now in 2026:\n\n---\n\n## \ud83e\udd1d 1. Multi-Agent Systems & Orchestration\nThe era of the single, all-purpose AI agent is giving way to **coordinat … [truncated]

event: message
data: {"subtype": "s

이제 **같은** `session_id`로 후속 요청을 보냅니다. 서버가 세션을 재개했기 때문에 에이전트가 첫 턴을 기억합니다:

In [7]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Tell me more about the second one."}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01YU5m6b66Kh6GcSN1Kbv7zq", "name": "WebSearch", "input": {"query": "context engineering AI agents 2026 techniques best practices"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, … [truncated]

[... 5 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "## \ud83e\uddf1 Deep Dive: Context Engineering\n\nContext engineering has quickly become **the defining AI skill of 2026**. Here's a thorough breakdown:\n\n---\n\n### What Is It, Exactly?\n\nContext engineering is the discipline of **designing what information an AI mode … [truncated]

event: message
data: {"subtype": "su

컨테이너를 재시작하고 후속 요청을 *하나 더* 보냅니다. 볼륨 마운트가 `/data`를 지켜 줬으므로 대화가 유지됩니다:

In [8]:
%%bash
cd hosting/docker && docker compose restart && sleep 3
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Summarize what we have discussed so far."}'

 Container docker-research-agent-1 Restarting 


 Container docker-research-agent-1 Started 


event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app

", "session_id": "77597263-a169-4236-b3d4-8ec14f90fd2b", "tools": ["Task", "TaskOutput", "Bash", "Gl

ob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "NotebookEdit", "WebFetch", "TodoWrite", "WebS

...

oid context rot and token blowouts.\n- **RAG alone isn't enough** \u2014 77% of IT leaders agree RAG

 is insufficient for production AI on its own.\n- Proper context engineering can improve agent task 

completion rates dramatically (e.g., **83% \u2192 96%**).\n\n---\n\nWould you like to explore any of

 these topics further?", "structured_output": null, "type": "ResultMessage"}



event: done
data: 



In [9]:
%%bash
# Teardown tier 1
cd hosting/docker && docker compose down

 Container docker-research-agent-1 Stopping 


 Container docker-research-agent-1 Stopped 
 Container docker-research-agent-1 Removing 


 Container docker-research-agent-1 Removed 


 Network docker_default Removing 


 Network docker_default Removed 


---
## 2단계 — Modal: 같은 이미지, 이제는 URL

1단계는 여러분의 머신에서 돌아갑니다. 2단계는 **같은 Dockerfile**을 [Modal](https://modal.com)의 `modal.Sandbox`로 실행해, 공개 HTTPS URL과 0까지 축소를 얻고 관리할 서버는 없습니다.

그 URL은 *공개*입니다. 주소를 아는 누구든 여러분의 API 예산을 쓸 수 있습니다. 1단계와 3단계는 앞단에 인증 게이트웨이가 있다고 가정하지만 2단계에는 게이트웨이가 없으므로, `modal_app.py`가 배포마다 베어러 토큰을 생성해 `AGENT_AUTH_TOKEN`으로 전달합니다. `server.py`는 그 환경 변수가 설정된 경우에만 토큰을 강제하므로 다른 단계는 영향을 받지 않습니다.

에이전트에 대해 바뀌는 것이 없으므로 [`hosting/modal/modal_app.py`](./hosting/modal/modal_app.py)는 짧습니다:

```python
app = modal.App.lookup("research-agent-hosting", create_if_missing=True)
image = modal.Image.from_dockerfile("hosting/Dockerfile", context_dir=".")
auth_token = secrets.token_urlsafe(32)
sandbox = modal.Sandbox.create(
    "serve",  # appended to the image's ENTRYPOINT, like compose's `command:`
    app=app,
    image=image,
    secrets=[
        modal.Secret.from_name("anthropic"),
        modal.Secret.from_dict({"AGENT_AUTH_TOKEN": auth_token}),
    ],
    volumes={"/data": sessions_volume},
    encrypted_ports=[8000],
)
print(sandbox.tunnels()[8000].url)
```

보존에는 `/data`에 마운트된 `modal.Volume`을 사용합니다. 1단계와 같은 `CLAUDE_CONFIG_DIR` 방식이죠. (샌드박스 여러 개가 동시에 쓰는 워크로드라 Volume 커밋 의미론 문제가 생긴다면 [`SessionStore`](https://code.claude.com/docs/en/agent-sdk/session-storage)로 바꾸세요. 3단계와 프로덕션 배포도 그것을 씁니다.)

일회성 설정은 **터미널에서** 하세요(`modal setup`이 브라우저를 열기 때문에 노트북 셀에서 실행할 수 없습니다):

```bash
pip install modal
modal setup
```

그런 다음 Modal이 `ANTHROPIC_API_KEY`로 주입할 시크릿을 만드세요:

In [10]:
%%bash
modal secret create anthropic ANTHROPIC_API_KEY="$(grep ANTHROPIC_API_KEY hosting/.env | cut -d= -f2)"

Created a new secret 'anthropic' with the key 'ANTHROPIC_API_KEY'



Use it in your Modal app:




@app.[38;2;248;248;242;48;

2;39;40;34mfunction(secret

s=[[38;2;248;248;242;4

...

                 
    os[

0m.getenv[38;2;248;248;242

;48;2;39;40;34m("ANTHROPIC

_API_KEY")[48;2;39;40

;34m                                              
                             

In [11]:
%%bash
python hosting/modal/modal_app.py | tee /tmp/modal_deploy.out
MODAL_URL=$(awk '/^url:/ {print $2}' /tmp/modal_deploy.out)
MODAL_TOKEN=$(awk '/^token:/ {print $2}' /tmp/modal_deploy.out)
{ echo "MODAL_URL=$MODAL_URL"; echo "MODAL_TOKEN=$MODAL_TOKEN"; } > /tmp/modal_url.env

sandbox: sb-7R7zQ7TtX0h9eKZ8qslvwo
url:     https://ta-01ks91e217n9fymaxjtdc9k5bh-8000-kn9n102ljd7y4

majwav00kwg0.w.modal.host
token:   sb-…redacted…

⚠️  The URL is p

ublic. The token is the only thing gating it — don't share both.

Try it:
  curl -N -X POST https:

//ta-01ks91e217n9fymaxjtdc9k5bh-8000-kn9n102ljd7y4majwav00kwg0.w.modal.host/sessions/demo-1/messages

 \
    -H 'Authorization: Bearer sb-…redacted…' \
    -H 'Content-Type

: application/json' \
    -d '{"prompt":"What are the latest AI agent trends?"}'


In [12]:
%%bash
source /tmp/modal_url.env
curl -N -s -X POST "$MODAL_URL/sessions/demo-1/messages" \
  -H "Authorization: Bearer $MODAL_TOKEN" \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Give me a one-sentence summary of the Claude Agent SDK."}'

event: message
data: {"subtype": "init", "data": {"type": "system", "subtype": "init", "cwd": "/app", "session_id": "1566ffe4-2b20-4a68-82ff-283984b64451", "tools": ["Task", "TaskOutput", "Bash", "Glob", "Grep", "ExitPlanMode", "Read", "Edit", "Write", "N … [truncated]

event: message
data: {"content": [{"id": "toolu_01PPS8yzMBzMnRhG2Hnpk5VL", "name": "WebSearch", "input": {"query": "Claude Agent SDK Anthropic 2026"}}], "model": "claude-sonnet-4-6", "parent_tool_use_id": null, "error": null, "usage": {"input_tokens": 120 … [truncated]

[... 5 events omitted — thinking blocks, tool loading, and web-search result payloads ...]

event: message
data: {"content": [{"text": "The **Claude Agent SDK** is Anthropic's framework that gives developers programmatic access to the same tools, agent loop, and context management that power Claude Code \u2014 enabling the creation of AI agents that can autonomously read files, run commands, search the w … [truncated]

event: message
data: {"subtype": "s

같은 인터페이스, 같은 이미지, 다른 호스트입니다. 아무도 호출하지 않을 때 Modal은 샌드박스를 0으로 축소하고 비용도 들지 않습니다.

유휴 리소스에 요금이 부과되지 않도록 정리합니다:

In [13]:
%%bash
python hosting/modal/teardown.py

terminating sandbox sb-7R7zQ7TtX0h9eKZ8qslvwo
deleted volume research-agent-sessions


---
## 3단계 — Kubernetes: 인프라를 직접 소유해야 할 때

3단계는 멀티테넌트 프로덕션, 규제 환경, 또는 네트워킹·격리·비용을 완전히 통제해야 하는 곳을 위한 것입니다. 에이전트 이미지와 인터페이스는 여전히 동일하고, 새로운 것은 그것을 *둘러싼* 장치입니다.

- 앞단의 **게이트웨이**가 호출자를 인증하고 그들이 소유한 `session_id`만 전달합니다. `server.py`가 남겨 둔 인증이 마침내 이뤄지는 자리입니다.
- **세션당 파드** 라우팅(게이트웨이 → Redis → 파드)으로 대화마다 자기 영향 범위를 갖습니다.
- 미리 데워 둔 파드의 **대기 풀**로 새 세션이 콜드 스타트 지연을 겪지 않게 합니다.
- **이그레스 잠금**(NetworkPolicy + 허용 목록 프록시)으로 프롬프트 인젝션을 당한 에이전트가 `api.anthropic.com` 외에는 어디에도 닿지 못하게 합니다.

전체 매니페스트, 게이트웨이, 단계별 아키텍처 설명은 [`hosting/kubernetes/`](./hosting/kubernetes/)에 있습니다. 클라우드 계정 없이 로컬 [kind](https://kind.sigs.k8s.io/) 클러스터에서 처음부터 끝까지 동작합니다:

```bash
cd hosting/kubernetes
ANTHROPIC_API_KEY=sk-ant-... ./kind-quickstart.sh
```

빠른 시작 스크립트는 데모 테넌트 두 명(`alice`와 `bob`)의 베어러 토큰을 출력합니다. 게이트웨이는 모든 세션을 그것을 만든 테넌트에 묶습니다. 1·2단계와 같은 `curl -N POST /sessions/{id}/messages`를 이제 `:8080`의 게이트웨이로 보내되 `Authorization: Bearer <alice-token>` 헤더를 붙이면 alice에게는 동작하고, 같은 요청에 bob의 토큰을 쓰면 403이 돌아옵니다. 실제 클러스터를 위한 레지스트리·인그레스 변경은 README의 *Deploying to your own cluster* 절에서 다룹니다.

*Kubernetes 단계는 Joe Shamon과 Ben Lehrburger의 내부 작업을 바탕으로 합니다.*

---
## 프로덕션 준비하기

각각 몇 줄이면 연결할 수 있는 프로덕션 고려 사항 두 가지입니다. 아래 셀에 코드가 있고, 전체 프로덕션 점검 목록(인증, 우아한 종료, 유휴 타임아웃 조정, 오토스케일링, 비용 통제)은 [호스팅 문서](https://code.claude.com/docs/en/agent-sdk/hosting)에서 다룹니다.

### 관측 가능성

SDK는 모든 턴과 도구 호출에 대해 OpenTelemetry 스팬을 내보냅니다. 환경 변수 두 개로 여러분의 컬렉터를 가리키기만 하면 되고, `server.py`는 전혀 고칠 필요가 없습니다([문서](https://code.claude.com/docs/en/agent-sdk/observability)):

In [14]:
# In docker-compose.yml / modal_app.py / your k8s Deployment:
#   OTEL_EXPORTER_OTLP_ENDPOINT=http://your-collector:4317
#   OTEL_SERVICE_NAME=research-agent

### 라이브니스

`GET /health`는 이미 `server.py`에 있습니다. 오케스트레이터의 라이브니스 프로브가 이를 가리키게 하세요(compose의 `healthcheck:`, Modal 헬스 체크, k8s `livenessProbe`).

### 볼륨을 넘어선 보존

`/data` 볼륨 마운트는 단일 호스트와 Modal에는 충분합니다. 멀티 호스트 프로덕션에서는 대화 기록을 공유 저장소(S3, Postgres, Redis)로 미러링하는 [`SessionStore` 어댑터](https://code.claude.com/docs/en/agent-sdk/session-storage)를 사용하세요. SessionStore는 미러라는 점에 유의하세요. 로컬 디스크 쓰기가 항상 먼저 일어나고, 미러 실패는 에이전트를 중단시키지 않고 `mirror_error`를 냅니다.

### 와이어 포맷

`server.py`는 원시 SDK 메시지 타입을 스트리밍합니다. 쿡북에는 충분하지만, 실제 API라면 SDK 버전이 올라가도 클라이언트가 깨지지 않도록 안정된 와이어 포맷을 정의하게 됩니다.

---
## 어떤 단계를 고를까

| 단계 | 얻는 것 | 이럴 때 고르세요 | 이럴 때 올라가세요 |
|---|---|---|---|
| **1. Docker** | 여러분이 통제하는 머신 위의 컨테이너. Compose가 재시작해 주고 바인드 마운트가 `/data`를 지킵니다. Docker 호스트 하나를 운영합니다. | 개발 루프, 사내 도구, 단일 테넌트 앱, 크론/배치 잡. 손으로 재시작할 수 있고 네트워크 밖에서는 아무도 호출하지 않습니다. | 그 머신 밖의 누군가에게 URL이 필요해지거나, "손으로 재시작"이 더는 받아들여지지 않을 때. |
| **2. Modal** | 같은 이미지를 공개 HTTPS URL 뒤에 두고 0까지 축소와 원격 빌드를 제공. 운영할 것이 없습니다. | 당장 호출 가능한 엔드포인트가 필요하고, 트래픽이 급증하거나 대부분 0이며, 배포별 베어러 토큰으로 인증이 충분할 때. | 진짜 멀티테넌트 격리나 네트워크 수준 이그레스 통제가 필요하거나, 플랫폼 팀이 워크로드를 자기 클러스터에 두도록 요구할 때. |
| **3. Kubernetes** | 세션당 파드 격리, 테넌트 범위 세션을 갖춘 인증 게이트웨이, 이그레스 잠금, 미리 데워 둔 대기 풀. 클러스터, 게이트웨이, Redis를 운영합니다. | 멀티테넌트 프로덕션, 규제 환경, 또는 이미 Kubernetes를 운영하고 있고 에이전트도 나머지와 같은 운영 모델을 따르기를 바랄 때. | 이 노트북 사다리의 꼭대기입니다. 여기서부터는 이전이 아니라 오토스케일링, 멀티 리전 라우팅, 영속 세션 저장소를 다듬게 됩니다. |

[호스팅 가이드](https://code.claude.com/docs/en/agent-sdk/hosting)의 배포 패턴이 이 단계들과 곧바로 대응됩니다. **일회성 세션**(패턴 1: 프롬프트 하나, 프로세스 하나, 종료)이 1a단계입니다. 서버 없이 크론이나 큐 워커에서 컨테이너를 실행하면 됩니다. **장기 실행**과 **하이브리드 세션**(패턴 2와 3, 프로세스가 턴에 걸쳐 대화 상태를 유지하거나 되살리는 형태)은 `server.py`가 `resume=`과 `/data` 볼륨으로 구현한 것입니다. 1b, 2, 3단계 모두 이 형태를 제공하며, 그 프로세스를 누가 살려 두고 호출자가 어떻게 닿느냐만 다릅니다. **단일 컨테이너**(패턴 4, 여러 세션을 컨테이너 하나에 다중화)가 바로 1b와 2단계가 하는 일입니다. 3단계는 그것이 더는 받아들여지지 않고 세션마다 자기 영향 범위가 필요할 때를 위해 존재합니다.

---
## 부록 — 다른 제공자로 옮기기

같은 `hosting/Dockerfile`, 다른 배포 명령입니다. 아래 모두 8000 포트를 노출하고 URL을 제공합니다. 보존을 위해 `/data`에 무언가를 마운트하세요.

**Fly Machines**
```bash
fly launch --dockerfile hosting/Dockerfile --no-deploy  # run from claude_agent_sdk/
fly volumes create data --size 1
fly deploy
```

**E2B**
```python
from e2b import Sandbox
sbx = Sandbox(template="research-agent")  # template built from hosting/Dockerfile
sbx.commands.run("./hosting/entrypoint.sh serve", background=True)
url = sbx.get_host(8000)
```

**Daytona**
```python
from daytona import Daytona, CreateSandboxFromImageParams
sbx = Daytona().create(CreateSandboxFromImageParams(image="research-agent"))
sbx.process.exec("./hosting/entrypoint.sh serve")
```

**Cloudflare Containers**
```ts
// wrangler.toml points at hosting/Dockerfile
export class Agent extends Container { defaultPort = 8000 }
```

**Vercel Sandbox**
```ts
import { Sandbox } from "@vercel/sandbox";
const sbx = await Sandbox.create({ image: "research-agent", ports: [8000] });
await sbx.runCommand({ cmd: "./hosting/entrypoint.sh", args: ["serve"], detached: true });
```